# Import des données socioéconmiques et des résultats électoraux issus des travaux de J. Cagé et T. Piketty

Import des bibliothèques

In [23]:
import pandas as pd
import numpy as np
import openpyxl
import glob
import os
import zipfile
import requests
import io
import matplotlib.pyplot as plt
import seaborn as sns
from zipfile import ZipFile
from io import BytesIO

## 1 : Import des données socioéconomiques

Import des données brutes

In [24]:
# On force le format texte pour les codes communes (évite les erreurs sur la Corse)
dtype_commune = {'codecommune': str}

# 1. Population
url_pop = "https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/Taille_agglo_commune_csv.zip"
content_pop = requests.get(url_pop).content
with zipfile.ZipFile(io.BytesIO(content_pop)) as z:
    with z.open("Taille_agglo_commune_csv/popcommunes.csv") as f:
        df_pop = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_1 : Population')

# 2. Âge et Sexe
url_agesex = "https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/Age_csp.zip"
content_agesex = requests.get(url_agesex).content
with zipfile.ZipFile(io.BytesIO(content_agesex)) as z:
    with z.open("Age_csp/agesexcommunes.csv") as f:
        df_age_sex = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_2 : Age/Sexe')

# 3. Étrangers
url_etrang = "https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/Nationalites_csv.zip"
content_etrang = requests.get(url_etrang).content
with zipfile.ZipFile(io.BytesIO(content_etrang)) as z:
    with z.open("Nationalites_csv/etrangerscommunes.csv") as f:
        df_etrang = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_3 : Étrangers')

# 4. Revenus
url_rev ="https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/Revenus_csv.zip"
content_rev = requests.get(url_rev).content
with zipfile.ZipFile(io.BytesIO(content_rev)) as z:
    with z.open("Revenus_csv/revcommunes.csv") as f:
        df_rev = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_4 : Revenus')

# 5. Diplômes
url_dipl ="https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/Diplomes_csv.zip"
content_dipl = requests.get(url_dipl).content
with zipfile.ZipFile(io.BytesIO(content_dipl)) as z:
    with z.open("Diplomes_csv/diplomescommunes.csv") as f:
        df_dipl = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_5 : Diplômes')

# 6. CSP
url_csp ="https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/CSP_csv.zip"
content_csp = requests.get(url_csp).content
with zipfile.ZipFile(io.BytesIO(content_csp)) as z: 
    with z.open("CSP_csv/cspcommunes.csv") as f:
        df_csp = pd.read_csv(f, low_memory=False, dtype=dtype_commune)
print('OK_import_6 : CSP')

OK_import_1 : Population
OK_import_2 : Age/Sexe
OK_import_3 : Étrangers
OK_import_4 : Revenus
OK_import_5 : Diplômes
OK_import_6 : CSP


Définitions des variables à récupérer

In [25]:
# On vient récupérer les années sur la période 1995-2022
liste_annees = range(1995,2023)

codecommune = ['codecommune']

# On définit l'ensemble des variables socioéconomiques que l'on souahite récupérer
var_pop = [f'pop{i}' for i in liste_annees]

var_age_sex = (
    [f'propf{i}' for i in liste_annees] + 
    [f'prop014{i}' for i in liste_annees] + 
    [f'prop1539{i}' for i in liste_annees] + 
    [f'prop4059{i}' for i in liste_annees] + 
    [f'prop60p{i}' for i in liste_annees]
)

var_etrang = [f'petranger{i}' for i in liste_annees]

var_csp = (
    [f'pcapi{i}' for i in liste_annees] + 
    [f'pouem{i}' for i in liste_annees] + 
    [f'paind{i}' for i in liste_annees] + 
    [f'pchom{i}' for i in liste_annees] 
)

var_dipl = (
    [f'pbac{i}' for i in liste_annees] + 
    [f'psup{i}' for i in liste_annees]
)

var_rev = [f'revmoy{i}' for i in liste_annees]

Séléction des données

In [26]:
df_pop = df_pop[codecommune+var_pop]
df_age_sex = df_age_sex[codecommune+var_age_sex]
df_etrang = df_etrang[codecommune+var_etrang]
df_csp = df_csp[codecommune+var_csp]
df_dipl = df_dipl[codecommune+var_dipl]
df_rev = df_rev[codecommune+var_rev]

Passage des DataFrame en format long

In [27]:
def pivot_df(df_large):
    # 1. On met le code commune en index pour ne pas le modifier
    df = df_large.set_index('codecommune')
    
    # 2. On coupe les noms de colonnes en deux (Ex: 'pop2007' -> ('pop', '2007'))
    df.columns = pd.MultiIndex.from_tuples(
        [(col[:-4], col[-4:]) for col in df.columns], 
        names=['Indicateur', 'Annee']
    )
    
    # 3. stack() descend le niveau 'Annee' en ligne 
    df_final = df.stack(level='Annee', future_stack=True).reset_index()

    # 4. On supprime le nom de l'axe des colonnes pour un affichage propre
    df_final.columns.name = None
    return df_final

In [28]:
df_pop_long = pivot_df(df_pop)
df_age_sex_long = pivot_df(df_age_sex)
df_etrang_long = pivot_df(df_etrang)
df_csp_long = pivot_df(df_csp)
df_dipl_long = pivot_df(df_dipl)
df_rev_long = pivot_df(df_rev)

Merge des données en un seul DataFrame

In [29]:
# 1. On standardise les clés de tous tes tableaux AVANT la fusion
liste_dfs = [
    df_pop_long, df_age_sex_long, df_etrang_long,
    df_csp_long, df_dipl_long, df_rev_long
]

for df in liste_dfs:
    # On s'assure que l'année est un entier
    df['Annee'] = df['Annee'].astype(int)
    # LE PLUS IMPORTANT : On force le code commune en texte à 5 caractères (remet les zéros !)
    df['codecommune'] = df['codecommune'].astype(str).str.strip().str.zfill(5)


# 2. On fait les fusions avec how='left' (pour ne garder QUE les communes/années du tableau pop)
df_socioeco = df_pop_long.copy()

df_socioeco = pd.merge(df_socioeco, df_age_sex_long, on=['codecommune', 'Annee'], how='left')
df_socioeco = pd.merge(df_socioeco, df_etrang_long, on=['codecommune', 'Annee'], how='left')
df_socioeco = pd.merge(df_socioeco, df_csp_long, on=['codecommune', 'Annee'], how='left')
df_socioeco = pd.merge(df_socioeco, df_dipl_long, on=['codecommune', 'Annee'], how='left')
df_socioeco = pd.merge(df_socioeco, df_rev_long, on=['codecommune', 'Annee'], how='left')

print(f"Lignes finales : {len(df_socioeco)}")

Lignes finales : 975968


## 2 Import des résultats électoraux

Import des données brutes

In [30]:
# 1. Fonction pour récupérer les résultats des législatives entre 1988 et 2022
def import_election (type_election, annee_election) :
    url = f"https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/{type_election}{annee_election}_csv.zip"
    print(url)
    content = requests.get(url).content
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        with z.open(f"{type_election}{annee_election}_csv/{type_election}{annee_election}comm.csv") as f:
            df= pd.read_csv(f, low_memory=False, dtype=dtype_commune)
    print('OK_import')
    return df

Sélection des élections

In [34]:
pres1995 = import_election ('pres',1995)
pres2002 = import_election ('pres',2002)
pres2007 = import_election ('pres',2007)
pres2012 = import_election ('pres',2012)
pres2017 = import_election ('pres',2017)
pres2022 = import_election ('pres',2022)

leg_1997 = import_election ('leg',1997)
leg_2002 = import_election ('leg',2002)
leg_2007 = import_election ('leg',2007)
leg_2012 = import_election ('leg',2012)
leg_2017 = import_election ('leg',2017)
leg_2022 = import_election ('leg',2022)

https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres1995_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres2002_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres2007_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres2012_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres2017_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/pres2022_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg1997_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg2002_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg2007_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg2012_csv.zip
OK_import
https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/z

Traitement des données électorales.

Récupération des résultats, par commune, des éléections legisltaives et présidentielles de 2007 à 2022 pour l'extreme droite et l'extreme gauche ainsi que l'abstention. 

In [35]:
#  TRAITEMENT ÉLECTORAL AVEC GAUCHE (VoteG) ---
cols_elec = ['codecommune', 'pabs']

# Présidentielles : On ajoute pvoteG (Gauche)
pres1995 = pres1995[cols_elec + ['pvoixLEPEN','pvoixJOSPIN']].rename(columns={'pvoixLEPEN': 'pvoix_RN', 'pvoixJOSPIN': 'vote_PS_pres'})
pres2002 = pres2002[cols_elec + ['pvoixLEPEN','pvoixJOSPIN']].rename(columns={'pvoixLEPEN': 'pvoix_RN', 'pvoixJOSPIN': 'vote_PS_pres'})
pres2007 = pres2007[cols_elec + ['pvoixLEPEN','pvoixROYAL']].rename(columns={'pvoixLEPEN': 'pvoix_RN', 'pvoixROYAL': 'vote_PS_pres'})
pres2012 = pres2012[cols_elec + ['pvoixMLEPEN','pvoixHOLLANDE']].rename(columns={'pvoixMLEPEN': 'pvoix_RN', 'pvoixHOLLANDE': 'vote_PS_pres'})
pres2017 = pres2017[cols_elec + ['pvoixMLEPEN','pvoixHAMON']].rename(columns={'pvoixMLEPEN': 'pvoix_RN', 'pvoixHAMON': 'vote_PS_pres'})
pres2022 = pres2022[cols_elec + ['pvoixMLEPEN','pvoixHIDALGO']].rename(columns={'pvoixMLEPEN': 'pvoix_RN', 'pvoixHIDALGO': 'vote_PS_pres'})



# Législatives : On ajoute pvoteG (Gauche)
leg_1997 = leg_1997[cols_elec + ['pvoixFN','pvoixPS']].rename(columns={'pvoixFN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})
leg_2002 = leg_2002[cols_elec + ['pvoixFN', 'pvoixPS']].rename(columns={'pvoixFN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})
leg_2007 = leg_2007[cols_elec + ['pvoixFN', 'pvoixPS']].rename(columns={'pvoixFN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})
leg_2012 = leg_2012[cols_elec + ['pvoixFN', 'pvoixPS']].rename(columns={'pvoixFN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})
leg_2017 = leg_2017[cols_elec + ['pvoixFN', 'pvoixPS']].rename(columns={'pvoixFN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})
leg_2022 = leg_2022[cols_elec + ['pvoixRN', 'pvoixNUP']].rename(columns={'pvoixRN': 'pvoix_RN', 'pvoixPS': 'vote_PS_leg'})

In [37]:
leg_2022 = leg_2022.rename(columns={'pvoixRN': 'pvoix_RN', 'pvoixNUP': 'vote_PS_leg'})

Passage en format long

In [38]:
for df, a in zip([pres1995, pres2002, pres2007, pres2012, pres2017, pres2022], [1995, 2002, 2007, 2012, 2017, 2022]): df['annee'] = a

pres_long = pd.concat([pres1995, pres2002, pres2007, pres2012, pres2017, pres2022], ignore_index=True)
pres_long = pres_long.rename(columns={'pvoix_RN': 'vote_RN_pres', 'pabs': 'abstention_pres'})

for df, a in zip([leg_1997, leg_2002, leg_2007, leg_2012, leg_2017, leg_2022], [1997, 2002, 2007, 2012, 2017, 2022]): df['annee'] = a

leg_long = pd.concat([leg_1997, leg_2002, leg_2007, leg_2012, leg_2017, leg_2022], ignore_index=True)
leg_long = leg_long.rename(columns={'pvoix_RN': 'vote_RN_leg', 'pabs': 'abstention_leg'})

In [39]:
pres_long

,codecommune,abstention_pres,vote_RN_pres,vote_PS_pres,annee
0,01001,0.201285,0.225352,0.247887,1995
1,01002,0.124260,0.251701,0.210884,1995
2,01004,0.242197,0.197880,0.232823,1995
3,01005,0.178117,0.266453,0.157303,1995
4,01006,0.188889,0.166667,0.194444,1995
...,...,...,...,...,...
216510,95676,0.128205,0.230769,0.008876,2022
216511,95678,0.119403,0.259188,0.007737,2022
216512,95680,0.323603,0.141969,0.008896,2022
216513,95682,0.127273,0.168421,0.021053,2022


Merge des données électorales

In [40]:
# UNIFICATION DES ÉLECTIONS 

electoral_final = pres_long.merge(
    leg_long, 
    on=["codecommune", "annee"], 
    how="outer"
)

electoral_final = electoral_final.rename(columns={'annee' : 'Annee'})
electoral_final.head(15)

,codecommune,abstention_pres,vote_RN_pres,vote_PS_pres,Annee,abstention_leg,vote_RN_leg,vote_PS_leg
0,.,0.214510,0.202756,0.010553,2022,NaN,NaN,NaN
1,01001,0.201285,0.225352,0.247887,1995,NaN,NaN,NaN
2,01001,NaN,NaN,NaN,1997,0.337607,0.200680,0.316327
3,01001,0.214920,0.192037,0.140515,2002,0.373239,0.000000,0.318644
4,01001,0.114094,0.118774,0.187739,2007,0.409699,0.058824,0.235294
5,01001,0.141892,0.252505,0.224449,2012,0.378151,0.195055,0.000000
6,01001,0.153846,0.254545,0.058586,2017,0.436455,0.181009,0.000000
7,01001,0.167442,0.286538,0.009615,2022,0.467391,0.256637,0.153392
8,01002,0.124260,0.251701,0.210884,1995,NaN,NaN,NaN
9,01002,NaN,NaN,NaN,1997,0.305389,0.263158,0.245614


Merge avec les données socioéconomiques

In [41]:
# 1. On force la colonne 'Annee' à être un nombre entier dans les deux DataFrames
df_socioeco['Annee'] = df_socioeco['Annee'].astype(int)
electoral_final['Annee'] = electoral_final['Annee'].astype(int)

# 2. Merging
données_caracteristiques_communes = df_socioeco.merge(electoral_final, on=["codecommune", "Annee"], how="left")

## 3 Import des typologies urbaines

Import des données

In [42]:
file_path = r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Typologie_urbaine\typo_urba_2021.xlsx" # Récupération des données concernant les typologies urbaines de l'INSEE
df_typo_urb = pd.read_excel(file_path, sheet_name='Maille communale', header=4)

# Adaptation des noms des colonnes
df_typo_urb.rename(columns={"CODGEO": "codecommune", "LIBDENS_AAV": "typologie_urbaine"}, inplace=True)

Mapping pour faire correspondre les données des deux bases. Cette typologie résulte d'un croisement entre le trio Rural, Urbain Intermédiare, Urbain Dense utilisé par l'INSEE dans ces données de typologie urbaine et les zonage AAV (de l'INSEE aussi). Cela permet de faire une disctinction majeure entre les communes rurales située à proximité d'une grande commune et celles qui ne le sont pas. Cela permettra d'affiner notre analyse.

In [43]:
# Création d' un dictionnaire de correspondance à partir du deuxième DataFrame
mapping = df_typo_urb.set_index('codecommune')['typologie_urbaine']

# Remplacement des valeurs dans le premier DataFrame
données_caracteristiques_communes['typologie_urbaine'] = données_caracteristiques_communes['codecommune'].map(mapping)

print(données_caracteristiques_communes['typologie_urbaine'].unique())

<ArrowStringArray>
['Rural non périurbain', 'Urbain intermédiaire',     'Rural périurbain',
         'Urbain dense',                    nan]
Length: 5, dtype: str


Gestion des valeurs manquantes

In [44]:
print(données_caracteristiques_communes['codecommune'].loc[données_caracteristiques_communes['typologie_urbaine'].isna()].unique())

<ArrowStringArray>
['75101', '75102', '75103', '75104', '75105', '75106', '75107', '75108',
 '75109', '75110', '75111', '75112', '75113', '75114', '75115', '75116',
 '75117', '75118', '75119', '75120']
Length: 20, dtype: str


In [45]:
liste = [f"{i}" for i in range (75101, 75121)] # liste des arrondissements parisiens

données_caracteristiques_communes.loc[données_caracteristiques_communes['codecommune'].isin(liste), 'typologie_urbaine'] = 'Urbain dense'

## Correction des données sur les bacheliers

In [46]:
# correction de la variable pbac
données_caracteristiques_communes['pbac'] = (données_caracteristiques_communes['pbac'] - données_caracteristiques_communes['psup']).clip(lower=0)

## Export des données

In [48]:
données_caracteristiques_communes.to_csv("données_caracteristiques_communes.csv")